# Clinical Scribe — Phase 1 (Free Colab T4)

Fine-tune **Qwen3-4B-Instruct-2507** with QLoRA to draft section-level clinical note text from doctor-patient dialogue.

> ⚠️ **Documentation-drafting aid, not a diagnostic tool.** Every output requires clinician review. Trained only on de-identified public benchmark data (MTS-Dialog, CC BY 4.0).

**This notebook is thin on purpose** — all logic lives in `src/clinical_scribe/`. It only: installs deps, mounts Drive, loads secrets, and calls the package via its CLI.

**Runtime:** set `Runtime → Change runtime type → T4 GPU` before running.

## 1. Get the code

In [ ]:
# Set to your repo URL (after you push it). Leave as-is if you've already
# uploaded/cloned the repo into the Colab working directory.
from google.colab import userdata
REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/Nitheesh555/clinical-scribe.git"
import os
if REPO_URL and not os.path.isdir("clinical-scribe"):
    !git clone $REPO_URL clinical-scribe
%cd clinical-scribe 2>/dev/null || echo "Assuming repo is the current directory"
!ls

## 2. Install dependencies
Torch + CUDA are preinstalled on Colab — do **not** reinstall torch. Unsloth pulls compatible transformers/trl/peft/bitsandbytes.

In [ ]:
%pip install -q unsloth
%pip install -q -e ".[eval]"
# Capture the exact resolved versions for reproducibility.
!pip freeze > requirements.lock
print("Wrote requirements.lock")

## 3. Mount Drive (crash-resilient checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/clinical-scribe/outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Checkpoints ->', DRIVE_DIR)

## 4. Secrets
Add `HF_TOKEN` (and optionally `WANDB_API_KEY`) in the Colab **🔑 Secrets** panel. They are exported to the environment — never hardcoded or printed.

In [ ]:
from google.colab import userdata
for key in ('HF_TOKEN', 'WANDB_API_KEY'):
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'{key}: set')
    except Exception:
        print(f'{key}: not provided (skipping)')

## 5. GPU check

In [ ]:
from clinical_scribe.utils import setup_logging, detect_gpu, log_versions
setup_logging()
detect_gpu()
log_versions()

## 6. Prepare data (MTS-Dialog → JSONL)

In [ ]:
!python scripts/prepare_data.py --config configs/phase1_t4.yaml --stats-out outputs/data_stats.json
import json; print(json.dumps(json.load(open('outputs/data_stats.json')), indent=2))

## 7. Train (QLoRA SFT)
Checkpoints to `general.checkpoint_dir` (set to Drive below). Re-run with `--resume` after a disconnect.

In [ ]:
# Point checkpoint mirroring at Drive without editing the YAML.
import yaml
with open('configs/_colab_override.yaml', 'w') as f:
    yaml.safe_dump({'extends': 'phase1_t4.yaml', 'general': {'checkpoint_dir': DRIVE_DIR}}, f)
!python -m clinical_scribe train --config configs/_colab_override.yaml  # add --resume to continue

## 8. Evaluate & export
Available once the eval/export steps are implemented (base-vs-fine-tuned ROUGE-L/BERTScore/faithfulness; merge + push to HF Hub + GGUF). Set `export.hub_repo_id` in the config first.

In [ ]:
# !python -m clinical_scribe evaluate --config configs/_colab_override.yaml --adapter outputs/adapter
# !python -m clinical_scribe export   --config configs/_colab_override.yaml --adapter outputs/adapter